# 02 — Backpropagation

This notebook explores the chain rule, numerical gradient checking, and backpropagation in a small neural network trained on a synthetic XOR dataset. It uses NumPy and Matplotlib only.

## 1. Setup

If needed, install dependencies with `python -m pip install numpy matplotlib`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=5, suppress=True)
SEED = 42
rng = np.random.default_rng(SEED)
print('NumPy version:', np.__version__)

## 2. A scalar chain-rule example

For `a = w*x + b`, `y = sigmoid(a)`, and `L = 0.5*(y-t)**2`, the chain rule gives `dL/dw = (y-t) * y*(1-y) * x` and `dL/db = (y-t) * y*(1-y)`.

In [ ]:
def sigmoid(z):
    # Clipping prevents overflow in exp for extreme values.
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-z))

x, w, b, target = 2.0, 0.5, -0.1, 1.0
a = w * x + b
y = sigmoid(a)
loss = 0.5 * (y - target) ** 2
dL_dy = y - target
dy_da = y * (1 - y)
dL_dw = dL_dy * dy_da * x
dL_db = dL_dy * dy_da
print(f'a={a:.5f}, prediction={y:.5f}, loss={loss:.5f}')
print(f'dL/dw={dL_dw:.5f}, dL/db={dL_db:.5f}')

## 3. Numerical gradient check

A central finite difference approximates a derivative by perturbing one parameter in both directions. It is a useful test for a hand-written backward pass.

In [ ]:
def scalar_loss(weight):
    prediction = sigmoid(weight * x + b)
    return 0.5 * (prediction - target) ** 2

eps = 1e-5
numerical_grad = (scalar_loss(w + eps) - scalar_loss(w - eps)) / (2 * eps)
print('Analytical gradient:', dL_dw)
print('Numerical gradient: ', numerical_grad)
print('Absolute difference:', abs(dL_dw - numerical_grad))
assert np.isclose(dL_dw, numerical_grad, rtol=1e-4, atol=1e-6)

## 4. Build a two-layer neural network

We will learn XOR. The hidden layer uses `tanh`; the output uses sigmoid and binary cross-entropy. For this combination, the output-layer error term simplifies to `(prediction - target) / number_of_examples`.

In [ ]:
# Four XOR examples; the model learns the nonlinear decision boundary.
X = np.array([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
y_true = np.array([[0.], [1.], [1.], [0.]])

def init_parameters(seed=7, hidden_units=4):
    r = np.random.default_rng(seed)
    return {
        'W1': r.normal(0, 0.8, size=(2, hidden_units)),
        'b1': np.zeros((1, hidden_units)),
        'W2': r.normal(0, 0.8, size=(hidden_units, 1)),
        'b2': np.zeros((1, 1)),
    }

def forward(X, params):
    z1 = X @ params['W1'] + params['b1']
    a1 = np.tanh(z1)
    z2 = a1 @ params['W2'] + params['b2']
    a2 = sigmoid(z2)
    cache = {'X': X, 'z1': z1, 'a1': a1, 'z2': z2, 'a2': a2}
    return a2, cache

def binary_cross_entropy(y, p):
    p = np.clip(p, 1e-9, 1 - 1e-9)
    return -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))

params = init_parameters()
pred, _ = forward(X, params)
print('Initial predictions:\n', pred)
print('Initial loss:', binary_cross_entropy(y_true, pred))

## 5. Backward pass and gradient descent

For `tanh`, the derivative is `1 - a1**2`. Gradients are averaged across the batch. All gradients are computed before any parameter is updated.

In [ ]:
def backward(y_true, params, cache):
    m = y_true.shape[0]
    a1, a2, Xb = cache['a1'], cache['a2'], cache['X']

    dz2 = (a2 - y_true) / m
    dW2 = a1.T @ dz2
    db2 = np.sum(dz2, axis=0, keepdims=True)

    da1 = dz2 @ params['W2'].T
    dz1 = da1 * (1 - a1 ** 2)
    dW1 = Xb.T @ dz1
    db1 = np.sum(dz1, axis=0, keepdims=True)
    return {'W1': dW1, 'b1': db1, 'W2': dW2, 'b2': db2}

learning_rate = 0.8
epochs = 12000
history = []
for epoch in range(epochs):
    pred, cache = forward(X, params)
    loss = binary_cross_entropy(y_true, pred)
    history.append(loss)
    grads = backward(y_true, params, cache)
    for key in params:
        params[key] -= learning_rate * grads[key]

pred, _ = forward(X, params)
print('Final loss:', binary_cross_entropy(y_true, pred))
print('Predictions:\n', pred)
print('Rounded classes:\n', (pred >= 0.5).astype(int))

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history)
plt.xlabel('Epoch')
plt.ylabel('Binary cross-entropy loss')
plt.title('Training loss on XOR')
plt.grid(True, alpha=0.3)
plt.show()

## 6. Check a network parameter with finite differences

We check one element of `W2` against the analytical gradient. This is a practical way to catch mistakes in a manual backward pass.

In [ ]:
# Use a fresh, small model so this check is independent of training.
check_params = init_parameters(seed=13, hidden_units=3)
check_pred, check_cache = forward(X, check_params)
check_grads = backward(y_true, check_params, check_cache)

def model_loss(p):
    prediction, _ = forward(X, p)
    return binary_cross_entropy(y_true, prediction)

key, idx = 'W2', (0, 0)
original = check_params[key][idx]
check_params[key][idx] = original + eps
loss_plus = model_loss(check_params)
check_params[key][idx] = original - eps
loss_minus = model_loss(check_params)
check_params[key][idx] = original
numerical = (loss_plus - loss_minus) / (2 * eps)
analytical = check_grads[key][idx]
print('Parameter:', key, idx)
print('Analytical:', analytical)
print('Numerical: ', numerical)
print('Absolute difference:', abs(analytical - numerical))
assert np.isclose(analytical, numerical, rtol=1e-4, atol=1e-6)

## 7. Takeaways

- Backpropagation computes gradients; gradient descent uses those gradients to update parameters.
- Intermediate activations are cached so the backward pass can reuse them.
- Matrix shapes are part of the algorithm: check them when debugging.
- Finite differences help verify manually derived gradients.

Next: complete `practice.md`, then review `solution.md`.